# 1. Data Preprocessing

## 1.1 Coordinate Reference System Transformation

This step reprojects Sentinel-1 Synthetic Aperture Radar (SAR)
imagery and corresponding iceberg polygon annotations into
the Antarctic Polar Stereographic coordinate reference system
(EPSG:3031), ensuring spatial consistency for subsequent
training sample preparation and iceberg segmentation.

In [ ]:

# ============================================================
# Reproject SAR imagery and iceberg annotations to EPSG:3031
# ============================================================

from pathlib import Path

import rasterio
import geopandas as gpd
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling,
)


# ------------------------------------------------------------
# 1. Input and output directories
# ------------------------------------------------------------

# Update these paths according to your local data structure.
BASE_DIR = Path(r"D:\CZL\Deep learning2")

SRC_DIR_TIF = BASE_DIR / "train_tif"
SRC_DIR_SHP = BASE_DIR / "label_shp"

OUT_DIR_TIF = BASE_DIR / "train_tif_3031"
OUT_DIR_SHP = BASE_DIR / "label_shp_3031"

TARGET_CRS = "EPSG:3031"
DEFAULT_VECTOR_CRS = "EPSG:4326"

# Create output directories if they do not exist.
OUT_DIR_TIF.mkdir(parents=True, exist_ok=True)
OUT_DIR_SHP.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. Reproject SAR GeoTIFF imagery
# ------------------------------------------------------------

raster_files = sorted(SRC_DIR_TIF.glob("*.tif"))

print(f"Found {len(raster_files)} SAR images.")

for tif_path in raster_files:

    output_path = OUT_DIR_TIF / tif_path.name

    print(f"[RASTER] Reprojecting: {tif_path.name}")

    with rasterio.open(tif_path) as src:

        if src.crs is None:
            raise ValueError(
                f"Missing coordinate reference system: {tif_path}"
            )

        # Calculate the output transform and raster dimensions.
        transform, width, height = calculate_default_transform(
            src.crs,
            TARGET_CRS,
            src.width,
            src.height,
            *src.bounds,
        )

        # Preserve the source metadata and update spatial properties.
        profile = src.profile.copy()

        profile.update({
            "crs": TARGET_CRS,
            "transform": transform,
            "width": width,
            "height": height,
            "compress": "lzw",
        })

        # Reproject the first raster band using bilinear resampling.
        with rasterio.open(output_path, "w", **profile) as dst:

            reproject(
                source=rasterio.band(src, 1),
                destination=rasterio.band(dst, 1),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=TARGET_CRS,
                resampling=Resampling.bilinear,
                src_nodata=src.nodata,
                dst_nodata=0.0,
            )

print(
    f"\n[COMPLETED] Raster reprojection: "
    f"{len(raster_files)} files.\n"
)


# ------------------------------------------------------------
# 3. Reproject iceberg polygon annotations
# ------------------------------------------------------------

vector_files = sorted(SRC_DIR_SHP.glob("*.shp"))

print(f"Found {len(vector_files)} annotation files.")

for shp_path in vector_files:

    output_path = OUT_DIR_SHP / shp_path.name

    print(f"[VECTOR] Reprojecting: {shp_path.name}")

    # Read iceberg polygon annotations.
    gdf = gpd.read_file(shp_path)

    # Assume EPSG:4326 if the source CRS is undefined.
    if gdf.crs is None:

        print(
            f"[WARNING] Missing CRS for {shp_path.name}. "
            f"Assuming {DEFAULT_VECTOR_CRS}."
        )

        gdf = gdf.set_crs(DEFAULT_VECTOR_CRS)

    # Transform vector geometries to EPSG:3031.
    gdf_3031 = gdf.to_crs(TARGET_CRS)

    # Save the reprojected annotations.
    gdf_3031.to_file(output_path)

print(
    f"\n[COMPLETED] Vector reprojection: "
    f"{len(vector_files)} files."
)

print("\n[SUCCESS] All reprojection tasks completed.")


## 1.2 Training Sample Generation for nnU-Net v2

This step converts preprocessed Sentinel-1 SAR GeoTIFF imagery and corresponding iceberg polygon annotations into training samples compatible with nnU-Net v2.

Positive samples are generated by expanding the bounding boxes of annotated icebergs with a 100-pixel margin, ensuring a minimum crop size of 256 × 256 pixels. All annotated icebergs intersecting each crop are included in the corresponding segmentation mask.

Negative samples are randomly extracted from background regions without overlapping annotated icebergs.

In [ ]:
# -*- coding: utf-8 -*-
# build_nnunet_from_sar_tiled_v3.py

import os
import glob
import json
from pathlib import Path
import numpy as np
import rasterio
from rasterio import features
from rasterio.windows import Window
from rasterio.warp import calculate_default_transform, reproject, Resampling
from shapely.geometry import box
from shapely.strtree import STRtree
import geopandas as gpd
import nibabel as nib
from tqdm import tqdm

# ======================== Configuration ========================
RAW_ROOT        = str(Path("nnUNet_raw_data"))
DATASET_ID      = 999
DATASET_NAME    = "Icebergs_Tiled"

TRAIN_TIF_DIR   = str(Path("train_tif_3031"))
TRAIN_VEC_DIR   = str(Path("label_shp_3031"))
TEST_TIF_DIR    = ""

TARGET_CRS      = None
TARGET_RES      = None  # Reserved; not applied by current raster reprojection.

ICE_SHELF_SHP   = ""  # Set to a valid coastline/ice-shelf exclusion polygon if needed.

PIXEL_PAD       = 100
MIN_TILE        = 256
MAX_TILE        = 20480000
ALL_TOUCHED     = True

NEG_PER_POS     = 3
NEG_TILE_SIZE   = 1024
NEG_TRY_LIMIT   = 100

MOD_NAME        = "SAR_HH"
PRINT_TOP_K     = 20
# =============================================================


def ensure_dir(p):
    """Create an output directory if it does not exist."""
    Path(p).mkdir(parents=True, exist_ok=True)


def list_tifs(folder):
    """Return sorted, unique GeoTIFF paths from an existing folder."""
    if not folder or not os.path.isdir(folder):
        return []
    files = {f for f in Path(folder).glob("*.tif")}
    files |= {f for f in Path(folder).glob("*.tiff")}
    return sorted(str(f) for f in files)


def pick_match(vec_dir, tif_stem):
    """Find the best matching Shapefile or GeoPackage for a raster stem."""
    shp = os.path.join(vec_dir, f"{tif_stem}.shp")
    gpkg = os.path.join(vec_dir, f"{tif_stem}.gpkg")
    if os.path.exists(shp): return shp
    if os.path.exists(gpkg): return gpkg
    cand = []
    for ext in ("*.shp", "*.gpkg"):
        for p in glob.glob(os.path.join(vec_dir, ext)):
            name = Path(p).stem
            if tif_stem == name or tif_stem in name or name in tif_stem:
                cand.append(p)
    if not cand: return None
    cand.sort(key=lambda p: -len(os.path.commonprefix([Path(p).stem, tif_stem])))
    return cand[0]


def _affine_to_nifti(transform):
    """Convert a Rasterio affine transform to a 4 x 4 NIfTI affine."""
    return np.array([
        [transform.a, transform.b, 0, transform.c],
        [transform.d, transform.e, 0, transform.f],
        [0,           0,           1, 0],
        [0,           0,           0, 1]
    ], dtype=np.float64)


def write_img_nifti(arr2d, transform, out_path):
    """Save a single-channel SAR image as a float32 NIfTI file."""
    nib.save(nib.Nifti1Image(arr2d.astype(np.float32, copy=False), _affine_to_nifti(transform)), out_path)


def write_lab_nifti(arr2d, transform, out_path):
    """Save a binary iceberg label as a uint16 NIfTI file."""
    nib.save(nib.Nifti1Image(arr2d.astype(np.uint16, copy=False), _affine_to_nifti(transform)), out_path)


def clamp_bounds(x0, y0, x1, y1, width, height):
    """Clip pixel bounds to the raster extent."""
    x0c = max(0, min(int(x0), width))
    y0c = max(0, min(int(y0), height))
    x1c = max(0, min(int(x1), width))
    y1c = max(0, min(int(y1), height))
    if x1c <= x0c or y1c <= y0c:
        return None
    return x0c, y0c, x1c, y1c


def crop_to_min_tile(x0, y0, x1, y1, width, height, min_tile):
    """Expand a crop to the requested minimum tile dimensions when possible."""
    w, h = x1 - x0, y1 - y0
    if w >= min_tile and h >= min_tile: return x0, y0, x1, y1
    cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
    half = min_tile // 2
    nx0, nx1 = int(cx - half), int(cx + half)
    ny0, ny1 = int(cy - half), int(cy + half)
    dx0, dy0 = 0 - nx0, 0 - ny0
    dx1, dy1 = nx1 - width, ny1 - height
    if dx0 > 0: nx0 += dx0; nx1 += dx0
    if dy0 > 0: ny0 += dy0; ny1 += dy0
    if dx1 > 0: nx0 -= dx1; nx1 -= dx1
    if dy1 > 0: ny0 -= dy1; ny1 -= dy1
    clamped = clamp_bounds(nx0, ny0, nx1, ny1, width, height)
    return clamped if clamped else (x0, y0, x1, y1)


def rasterize_mask(gdf, out_shape, transform, target_crs, all_touched=True):
    """Rasterize valid vector geometries onto a target image grid."""
    if gdf.empty: return np.zeros(out_shape, dtype=np.uint8)
    if gdf.crs is None: raise ValueError("Input vector data has no coordinate reference system (CRS).")
    if str(gdf.crs) != str(target_crs): gdf = gdf.to_crs(target_crs)
    gdf = gdf[gdf.is_valid & ~gdf.is_empty]
    shapes = [(geom, 1) for geom in gdf.geometry if geom.is_valid]
    if not shapes: return np.zeros(out_shape, dtype=np.uint8)
    return features.rasterize(
        shapes=shapes,
        out_shape=out_shape,
        transform=transform,
        fill=0,
        all_touched=all_touched,
        dtype=np.uint8
    )


def _open_raster_as_target(src_ds, target_crs, target_res=None):
    """Read band one, optionally reprojecting to the target CRS."""
    src_crs = src_ds.crs
    src_transform = src_ds.transform
    src_nodata = src_ds.nodata
    width, height = src_ds.width, src_ds.height

    if not target_crs or (src_crs and str(src_crs) == str(target_crs)):
        arr = src_ds.read(1).astype(np.float32)
        if src_nodata is not None:
            arr = np.where(arr == src_nodata, np.nan, arr)
        arr = np.nan_to_num(arr, nan=0.0)
        return arr, src_transform, src_crs

    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, target_crs, width, height, *src_ds.bounds
    )
    dst_arr = np.zeros((dst_height, dst_width), dtype=np.float32)
    reproject(
        source=rasterio.band(src_ds, 1),
        destination=dst_arr,
        src_transform=src_transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=target_crs,
        resampling=Resampling.bilinear,
        src_nodata=src_nodata,
        dst_nodata=np.nan
    )
    dst_arr = np.nan_to_num(dst_arr, nan=0.0)
    return dst_arr, dst_transform, rasterio.crs.CRS.from_string(target_crs)


def strtree_query(tree, geoms_list, query_geom):
    """Return candidate geometries using Shapely STRtree results."""
    hits = tree.query(query_geom)
    if hits is None:
        return []
    if isinstance(hits, (list, tuple)):
        if not hits:
            return []
        if hasattr(hits[0], "geom_type"):
            return list(hits)
        idxs = np.asarray(hits).ravel().astype(int)
        return [geoms_list[i] for i in idxs]
    if np.isscalar(hits):
        idxs = np.array([int(hits)], dtype=int)
    else:
        idxs = np.asarray(hits).ravel().astype(int)
    return [geoms_list[i] for i in idxs]


def process_one_pair(tif_path, vec_path, imagesTr, labelsTr, id_prefix):
    """Generate iceberg-positive and background-negative training patches."""
    n_out = 0
    with rasterio.open(tif_path) as src:
        img, transform, crs = _open_raster_as_target(src, TARGET_CRS, TARGET_RES)
        H, W = img.shape

        # Apply the optional geographic exclusion mask to image intensities.
        if ICE_SHELF_SHP and os.path.exists(ICE_SHELF_SHP):
            try:
                shelf_gdf = gpd.read_file(ICE_SHELF_SHP)
                if shelf_gdf.crs is None:
                    shelf_gdf = shelf_gdf.set_crs("EPSG:4326")
                if str(shelf_gdf.crs) != str(crs):
                    shelf_gdf = shelf_gdf.to_crs(crs)
                shelf_mask = features.rasterize(
                    [(geom, 1) for geom in shelf_gdf.geometry if geom.is_valid],
                    out_shape=(H, W),
                    transform=transform,
                    fill=0,
                    all_touched=True,
                    dtype=np.uint8
                )
                img = np.where(shelf_mask == 1, np.nan, img)
                img = np.nan_to_num(img, nan=0.0)
            except Exception as e:
                print(f"[WARN] Failed to apply exclusion mask: {e}")

        # Read and spatially index valid iceberg annotations.
        gdf = gpd.read_file(vec_path)
        if gdf.empty or "geometry" not in gdf.columns:
            geoms, tree = [], None
        else:
            if gdf.crs is None:
                gdf = gdf.set_crs(crs)
            elif str(gdf.crs) != str(crs):
                gdf = gdf.to_crs(crs)
            valid_geoms = []
            for g in gdf.geometry:
                if g is None:
                    continue
                try:
                    if g.is_valid and not g.is_empty:
                        valid_geoms.append(g)
                except Exception:
                    continue
            geoms = valid_geoms
            tree = STRtree(geoms) if geoms else None

        # Generate positive crops around individual iceberg annotations.
        if geoms:
            for k, geom in enumerate(geoms):
                minx, miny, maxx, maxy = geom.bounds
                c0, r0 = ~transform * (minx, maxy)
                c1, r1 = ~transform * (maxx, miny)
                y0, y1 = sorted([int(np.floor(r0)), int(np.ceil(r1))])
                x0, x1 = sorted([int(np.floor(c0)), int(np.ceil(c1))])
                x0 -= PIXEL_PAD; y0 -= PIXEL_PAD
                x1 += PIXEL_PAD; y1 += PIXEL_PAD
                clipped = clamp_bounds(x0, y0, x1, y1, W, H)
                if not clipped: continue
                x0c, y0c, x1c, y1c = crop_to_min_tile(*clipped, W, H, MIN_TILE)
                x1c = min(x1c, x0c + MAX_TILE)
                y1c = min(y1c, y0c + MAX_TILE)
                tile = img[y0c:y1c, x0c:x1c]
                tile_transform = rasterio.windows.transform(
                    Window(x0c, y0c, x1c - x0c, y1c - y0c), transform
                )
                if tree is not None:
                    tile_box = box(*rasterio.windows.bounds(Window(x0c, y0c, x1c-x0c, y1c-y0c), transform))
                    cand = strtree_query(tree, geoms, tile_box)
                    cand = [g for g in cand if g.intersects(tile_box)]
                    sub_gdf = gpd.GeoDataFrame(geometry=cand, crs=crs)
                else:
                    sub_gdf = gpd.GeoDataFrame(geometry=[], crs=crs)
                mask = rasterize_mask(sub_gdf, tile.shape, tile_transform, crs, all_touched=ALL_TOUCHED)
                if np.count_nonzero(mask) == 0:
                    continue
                stem = f"{id_prefix}_pos_{k:04d}"
                write_img_nifti(tile, tile_transform, str(imagesTr / f"{stem}_0000.nii.gz"))
                write_lab_nifti(mask, tile_transform, str(labelsTr / f"{stem}.nii.gz"))
                n_out += 1

        # Sample negative crops without overlaps with annotated icebergs.
        if NEG_PER_POS > 0:
            # A fixed attempt cap may prevent achieving the requested ratio.
            need = (len(geoms) if geoms else 0) * NEG_PER_POS
            rng = np.random.default_rng(12345)
            made, tries = 0, 0
            while made < need and tries < NEG_TRY_LIMIT:
                tries += 1
                if W < NEG_TILE_SIZE or H < NEG_TILE_SIZE: break
                x0 = int(rng.integers(0, max(1, W - NEG_TILE_SIZE)))
                y0 = int(rng.integers(0, max(1, H - NEG_TILE_SIZE)))
                x1, y1 = x0 + NEG_TILE_SIZE, y0 + NEG_TILE_SIZE
                win = Window(x0, y0, NEG_TILE_SIZE, NEG_TILE_SIZE)
                tile_box = box(*rasterio.windows.bounds(win, transform))
                if geoms and tree is not None:
                    hits = strtree_query(tree, geoms, tile_box)
                    if any(g.intersects(tile_box) for g in hits):
                        continue
                tile = img[y0:y1, x0:x1]
                tile_transform = rasterio.windows.transform(win, transform)
                mask = np.zeros(tile.shape, dtype=np.uint8)
                stem = f"{id_prefix}_neg_{made:04d}"
                write_img_nifti(tile, tile_transform, str(imagesTr / f"{stem}_0000.nii.gz"))
                write_lab_nifti(mask, tile_transform, str(labelsTr / f"{stem}.nii.gz"))
                made += 1
                n_out += 1
            if made < need:
                print(f"[WARN] Generated {made}/{need} negative samples for {id_prefix}. "
                      f"The attempt limit ({NEG_TRY_LIMIT}) was reached or the scene was too small.")
    return n_out


def _convert_full_image_to_nifti(tif_path, out_dir):
    """Export a full single-band SAR scene as a NIfTI test image."""
    with rasterio.open(tif_path) as src:
        arr, transform, _ = _open_raster_as_target(src, TARGET_CRS, TARGET_RES)
    write_img_nifti(arr, transform, str(Path(out_dir) / f"{Path(tif_path).stem}_0000.nii.gz"))


def _print_top_k_samples(imagesTr, labelsTr, k=20):
    """Print basic pixel and label statistics for saved training samples."""
    img_list = sorted(Path(imagesTr).glob("*_0000.nii.gz"))
    lab_list = sorted(Path(labelsTr).glob("*.nii.gz"))
    n_show = min(k, len(img_list), len(lab_list))
    print(f"\n=== Preview of the first {n_show} training samples ===\n")
    for i in range(n_show):
        ip, lp = img_list[i], lab_list[i]
        img, lab = nib.load(str(ip)).get_fdata(), nib.load(str(lp)).get_fdata()
        iceberg_pixels = int(np.count_nonzero(lab))
        iceberg_ratio = iceberg_pixels / lab.size * 100 if lab.size else 0.0
        print(f"[{i+1:02d}] {ip.stem}")
        print(f"   Shape: {img.shape}, range: {img.min():.2f}–{img.max():.2f}, mean: {img.mean():.2f}")
        print(f"   Label: {iceberg_pixels} iceberg pixels ({iceberg_ratio:.3f}%)\n")


def main():
    """Build the nnU-Net dataset and write dataset metadata."""
    ds_dir = Path(RAW_ROOT) / f"Dataset{DATASET_ID}_{DATASET_NAME}"
    imagesTr, labelsTr, imagesTs = ds_dir / "imagesTr", ds_dir / "labelsTr", ds_dir / "imagesTs"
    for d in (imagesTr, labelsTr, imagesTs): ensure_dir(d)
    tifs = list_tifs(TRAIN_TIF_DIR)
    if not tifs: raise RuntimeError(f"No training images found in: {TRAIN_TIF_DIR}")
    print(f"[INFO] Found {len(tifs)} training images.")
    n_train = 0
    for tif_path in tqdm(tifs, desc="Processing training images"):
        stem = Path(tif_path).stem
        vec = pick_match(TRAIN_VEC_DIR, stem)
        if vec is None:
            print(f"[WARN] No matching annotation for {stem}; skipping.")
            continue
        n_train += process_one_pair(tif_path, vec, imagesTr, labelsTr, stem)
    if TEST_TIF_DIR and os.path.isdir(TEST_TIF_DIR):
        for tif_path in tqdm(list_tifs(TEST_TIF_DIR), desc="Converting test images"):
            _convert_full_image_to_nifti(tif_path, imagesTs)
    ds_json = {
        "name": f"Dataset{DATASET_ID}_{DATASET_NAME}",
        "description": "SAR iceberg tiles for nnU-Net v2",
        "tensorImageSize": "2D",
        "channel_names": {"0": MOD_NAME},
        "labels": {"background": 0, "iceberg": 1},
        "numTraining": int(n_train),
        "file_ending": ".nii.gz"
    }
    with open(ds_dir / "dataset.json", "w", encoding="utf-8") as f:
        json.dump(ds_json, f, indent=2, ensure_ascii=False)
    print(f"\n[COMPLETED] Dataset: {ds_dir}\nTraining samples: {n_train}")
    try:
        _print_top_k_samples(imagesTr, labelsTr, PRINT_TOP_K)
    except Exception as e:
        print(f"[WARN] Failed to preview samples: {e}")


if __name__ == "__main__":
    main()


# 2. Model Training

## 2.1 nnU-Net v2 Training and Monitoring

This step prepares the iceberg segmentation dataset for nnU-Net v2, performs dataset integrity checks, extracts dataset fingerprints, generates experiment plans, and trains the segmentation model with progress monitoring.

In [ ]:
"""nnU-Net v2 dataset checks, planning, training, and metric monitoring.

Configure the dataset and storage paths before running. This pipeline may
modify dataset.json and label filenames; back up training data first.
"""

import os, re, time, subprocess, sys, csv, json, glob, shutil, math
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib
# Select a non-interactive backend on headless Linux systems.
if os.name != "nt" and not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict
from typing import Optional
from shutil import which

# ---------------- User configuration ----------------
DATASET_ID   = 999
DATASET_NAME = "Icebergs_Tiled"

PROJECT_ROOT = Path(os.environ.get("ICEBERG_PROJECT_DIR", str(Path.cwd())))
RAW_ROOT     = str(PROJECT_ROOT / "nnUNet_raw_data")
PRE_ROOT     = str(PROJECT_ROOT / "nnUNet_preprocessed")
RES_ROOT     = str(PROJECT_ROOT / "nnUNet_results")

PLANNER      = "nnUNetPlannerResEncM"   # Residual-encoder medium preset; verify support in your nnU-Net version.
CONFIG       = "2d"                     # Example configurations: 2d, 3d_fullres, 3d_lowres.
FOLDS        = [0]                      # Fold 0 only; use [0, 1, 2, 3, 4] for five-fold training.

# Minimum free space and Windows path-length warning threshold.
MIN_FREE_GB_PRE = 20
MIN_FREE_GB_RES = 10
LONG_PATH_WARN  = 230

# Export training monitoring figures after training.
SAVE_FIG = True

# Set nnU-Net environment variables.
os.environ["nnUNet_raw"]          = RAW_ROOT
os.environ["nnUNet_preprocessed"] = PRE_ROOT
os.environ["nnUNet_results"]      = RES_ROOT

# ---------------- Filesystem utilities ----------------
def _bytes_to_gb(x: int) -> float:
    return round(x / (1024**3), 2)

def _check_disk_free(path: str, min_gb: float, tag: str):
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    total, used, free = shutil.disk_usage(str(p))
    if free / (1024**3) < min_gb:
        raise RuntimeError(f"{tag} Insufficient free space: {_bytes_to_gb(free)} GB < {min_gb} GB, path:{p}")
    print(f"[check] {tag} free space:{_bytes_to_gb(free)} GB")

def _warn_long_paths(root: Path, limit: int = LONG_PATH_WARN):
    bad = []
    for rp, _, files in os.walk(root):
        for fn in files:
            full = str(Path(rp)/fn)
            if len(full) >= limit:
                bad.append(full)
    if bad:
        print(f"[warn] Found {len(bad)} files that may exceed Windows path limits (>{limit}characters).Example:\n  {bad[0]}")
    else:
        print("[check] No suspiciously long paths found.")

# ---------------- Command-line invocation ----------------
SCRIPTS = Path(sys.executable).parent / "Scripts"

def _exe_path(name: str):
    cand = SCRIPTS / (name + (".EXE" if os.name == "nt" else ""))
    return str(cand) if cand.exists() else None

def run_cmd_pref(name: str, args: str, module_fallback: str = None):
    """
    Invocation order:
      1) Scripts/.exe
      2) command on PATH
      3) python -m nnunetv2.experiment_planning.plan_and_preprocess
      4) python -m nnunetv2.experiment_planning.plan_and_preprocess_entrypoints
      5) explicit fallback
    Command failures include captured stdout/stderr for diagnosis.
    """
    def _run(cmd: str, tag: str):
        print(f"[RUN:{tag}] {cmd}")
        res = subprocess.run(cmd, shell=True, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if res.returncode != 0:
            print(f"[{tag}] returncode={res.returncode}")
            if res.stdout:
                print(f"[{tag}] ---- begin captured output ----\n{res.stdout}\n[{tag}] ---- end captured output ----")
        else:
            if res.stdout:
                print(f"[{tag}] ---- output ----\n{res.stdout}\n[{tag}] ---- output end ----")
        return res

    # 1) Executable in the active Python environment.
    exe = _exe_path(name)
    if exe:
        res = _run(f'"{exe}" {args}', f"{name}.exe")
        if res.returncode == 0:
            return

    # 2) Executable found on PATH.
    if which(name):
        res = _run(f'{name} {args}', f"{name}@PATH")
        if res.returncode == 0:
            return

    # 3) Attempt module-based entry point.
    guess_mod = "nnunetv2.experiment_planning.plan_and_preprocess"
    res = _run(f'"{sys.executable}" -u -m {guess_mod} {args}', f"-m {guess_mod}")
    if res.returncode == 0:
        return

    # 4) Attempt alternate module entry point.
    alt_mod = "nnunetv2.experiment_planning.plan_and_preprocess_entrypoints"
    res = _run(f'"{sys.executable}" -u -m {alt_mod} {args}', f"-m {alt_mod}")
    if res.returncode == 0:
        return

    # 5) Explicit fallback module.
    if module_fallback and module_fallback not in (guess_mod, alt_mod):
        res = _run(f'"{sys.executable}" -u -m {module_fallback} {args}', f"-m {module_fallback}")
        if res.returncode == 0:
            return

    raise RuntimeError(f"{name} failed after executable and module-based attempts; see output above.")

# ---------------- Training metrics and monitoring ----------------
@dataclass
class MetricRow:
    time_s: float
    epoch: Optional[int] = None
    max_epoch: Optional[int] = None
    iter: Optional[int] = None
    train_loss: Optional[float] = None
    val_loss: Optional[float] = None
    dice: Optional[float] = None
    lr: Optional[float] = None
    raw: str = ""

RE_EPOCHOF = re.compile(r"(?:epoch|epo)\s*[:= ]?\s*(\d+)\s*/\s*(\d+)", re.I)
RE_EPOCH   = re.compile(r"(?:epoch|epo)\s*[:= ]?\s*(\d+)", re.I)
RE_ITER    = re.compile(r"(?:iter|it)\s*[:= ]?\s*(\d+)", re.I)

# Match validation loss, training loss, and a generic loss fallback.
_PAT = {
    "val_loss": [re.compile(r"(?:val(?:idation)?[_\s-]*loss)\s*[:= ]\s*(-?[0-9]*\.?[0-9]+)", re.I)],
    "train_loss": [
        re.compile(r"(?:train(?:ing)?[_\s-]*loss)\s*[:= ]\s*(-?[0-9]*\.?[0-9]+)", re.I),
        re.compile(r"\bloss\s*[:= ]\s*(-?[0-9]*\.?[0-9]+)", re.I),
    ],
    "dice": [re.compile(r"(?:mean[_\s-]*fg[_\s-]*dice|fg[_\s-]*dice|mean[_\s-]*dice|dice)\s*[:= ]\s*(-?[0-9]*\.?[0-9]+)", re.I)],
    "lr":   [re.compile(r"(?:lr|learning[_\s-]*rate)\s*[:= ]\s*([0-9]*\.?[0-9eE+-]+)", re.I)],
}

def _parse_line(line: str, t0: float) -> MetricRow:
    now = time.time()
    row = MetricRow(time_s=now - t0, raw=line.strip())

    m = RE_EPOCHOF.search(line)
    if m:
        row.epoch, row.max_epoch = int(m.group(1)), int(m.group(2))
    else:
        m = RE_EPOCH.search(line)
        if m: row.epoch = int(m.group(1))

    m = RE_ITER.search(line)
    if m: row.iter = int(m.group(1))

    for k, regs in _PAT.items():
        for r in regs:
            m = r.search(line)
            if m:
                try:
                    val = float(m.group(1))
                except Exception:
                    continue
                if k == "dice" and val > 1.0:
                    val = val / 100.0
                setattr(row, k, val)
                break
    return row

class _LivePlot:
    def __init__(self, title="nnU-Net Training Monitor", throttle_every=30):
        self.t, self.tr, self.va, self.di = [], [], [], []
        self.fig, (self.ax1, self.ax2) = plt.subplots(2, 1, figsize=(8,7))
        self.fig.suptitle(title)
        (self.l1,) = self.ax1.plot([], [], label="Train Loss")
        (self.l2,) = self.ax1.plot([], [], label="Val Loss")
        self.ax1.set_xlabel("Time (s)")
        self.ax1.set_ylabel("Loss"); self.ax1.legend(); self.ax1.grid(True, ls="--", alpha=0.3)
        (self.l3,) = self.ax2.plot([], [], label="Dice")
        self.ax2.set_xlabel("Time (s)")
        self.ax2.set_ylabel("Dice"); self.ax2.set_ylim(0,1); self.ax2.legend(); self.ax2.grid(True, ls="--", alpha=0.3)
        plt.tight_layout(rect=[0,0,1,0.96])
        try: plt.show(block=False)
        except Exception: pass
        self._throttle_every = throttle_every

    def _fill(self, seq):
        out, last = [], None
        for v in seq:
            if v is None:
                out.append(last if last is not None else np.nan)
            else:
                last=v; out.append(v)
        return out

    def update(self, row: MetricRow):
        if not any([row.train_loss, row.val_loss, row.dice]): return
        self.t.append(row.time_s)
        self.tr.append(row.train_loss if row.train_loss is not None else (self.tr[-1] if self.tr else None))
        self.va.append(row.val_loss if row.val_loss is not None else (self.va[-1] if self.va else None))
        d = row.dice
        if d is not None and d > 1: d = d / 100.0
        self.di.append(d if d is not None else (self.di[-1] if self.di else None))
        if len(self.t) % self._throttle_every != 0:
            return
        tr, va, di = self._fill(self.tr), self._fill(self.va), self._fill(self.di)
        self.l1.set_data(self.t, tr); self.l2.set_data(self.t, va)
        self.ax1.relim(); self.ax1.autoscale_view()
        self.l3.set_data(self.t, di)
        self.ax2.relim(); self.ax2.autoscale_view(scalex=True, scaley=False)
        try: plt.pause(0.001)
        except Exception: pass

    def save_png(self, out_png: Path):
        try:
            self.fig.savefig(out_png, dpi=140)
            print(f"[monitor] Training plot saved: {out_png}")
        except Exception as e:
            print(f"[warn] Failed to save training plot: {e}")

def monitor_training(run_cmd=None, csv_out="nnunet_metrics.csv", title="nnU-Net Training Monitor", poll=0.2, quiet=False):
    assert (run_cmd is not None), "A training command must be provided"
    plotter = _LivePlot(title)
    t0 = time.time()
    fcsv = open(csv_out, "w", newline="", encoding="utf-8")
    w = csv.DictWriter(fcsv, fieldnames=list(asdict(MetricRow(0)).keys()))
    w.writeheader()

    def feed_text(text, carry):
        text = (carry[0] + text).replace("\r", "\n")
        parts = text.split("\n")
        carry[0] = parts[-1]
        return parts[:-1]

    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    patched = run_cmd
    if re.match(r"^\s*python(\.exe)?\s+", run_cmd, re.I) and " -u " not in run_cmd:
        patched = run_cmd.replace("python", "python -u", 1)

    proc = subprocess.Popen(patched, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0, env=env)
    carry = [""]

    try:
        while True:
            chunk = proc.stdout.read(4096)
            if not chunk:
                tail = carry[0]
                if tail:
                    for line in feed_text("\n", carry):
                        if not quiet: print(line)
                        row = _parse_line(line, t0)
                        if any([row.train_loss, row.val_loss, row.dice, row.epoch, row.iter, row.lr]):
                            w.writerow(asdict(row)); plotter.update(row)
                    carry[0] = ""
                if proc.poll() is not None: break
                time.sleep(poll); continue
            try:
                enc = sys.stdout.encoding or "utf-8"
                text = chunk.decode(enc, errors="replace")
            except Exception:
                text = chunk.decode("utf-8", errors="replace")
            for line in feed_text(text, carry):
                if not quiet: print(line)
                row = _parse_line(line, t0)
                if any([row.train_loss, row.val_loss, row.dice, row.epoch, row.iter, row.lr]):
                    w.writerow(asdict(row)); plotter.update(row)
            try: plt.pause(0.001)
            except Exception: pass
    finally:
        try: fcsv.flush(); fcsv.close()
        except Exception: pass
        try:
            if proc and proc.stdout: proc.stdout.close()
        except Exception: pass

    if SAVE_FIG:
        png_path = Path(csv_out).with_suffix(".png")
        plotter.save_png(png_path)

    print(f"Monitoring complete; CSV written to: {csv_out}")
    if proc.returncode != 0:
        raise RuntimeError(f"Training command failed with exit code {proc.returncode}")

# ---------------- Dataset checks and optional repairs ----------------
ds_dir   = Path(RAW_ROOT) / f"Dataset{DATASET_ID}_{DATASET_NAME}"
imagesTr = ds_dir / "imagesTr"
labelsTr = ds_dir / "labelsTr"
js_path  = ds_dir / "dataset.json"

assert ds_dir.exists(), f"Dataset directory does not exist: {ds_dir}"
assert js_path.exists(), f"Missing dataset.json: {js_path}"

with open(js_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

if "channel_names" not in meta:
    meta["channel_names"] = {"0": meta.get("modality", {}).get("0", "SAR_HH")}
if "file_ending" not in meta:
    meta["file_ending"] = ".nii.gz"
if "labels" not in meta or all(str(k).isdigit() for k in meta["labels"]):
    meta["labels"] = {"background": 0, "iceberg": 1}

with open(js_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print("dataset.json updated: ", meta)

_check_disk_free(PRE_ROOT, MIN_FREE_GB_PRE, "Preprocessed data directory")
_check_disk_free(RES_ROOT, MIN_FREE_GB_RES, "Results directory")
_warn_long_paths(ds_dir, LONG_PATH_WARN)

def quick_dataset_sanity(imagesTr: Path, labelsTr: Path, js_path: Path):
    print("\n[check] Quick dataset validation ...")
    with open(js_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    need = ["labels", "channel_names", "file_ending"]
    miss = [k for k in need if k not in meta]
    if miss:
        print(f"[warn] dataset.json Missing fields: {miss}")
    imgs = sorted([p for p in imagesTr.glob("*.nii.gz") if p.name.endswith("_0000.nii.gz")])
    lbls = sorted(labelsTr.glob("*.nii.gz"))
    print(f"[check] imagesTr={len(imgs)}  labelsTr={len(lbls)}")
    bad_pairs, empty_labels = [], []
    for img in imgs:
        cid = img.name[:-len("_0000.nii.gz")]
        lab = labelsTr / f"{cid}.nii.gz"
        if not lab.exists():
            bad_pairs.append((img.name, f"{cid}.nii.gz"))
        else:
            try:
                arr = nib.load(str(lab)).get_fdata()
                if float(np.max(arr)) == 0.0:
                    empty_labels.append(lab.name)
            except Exception as e:
                print(f"[warn] Failed to read label {lab.name}: {e}")
    if bad_pairs:
        print(f"[fatal] {len(bad_pairs)} image cases are missing labels, example: {bad_pairs[0]}")
    # Empty labels are valid for background-only training cases.
    if empty_labels:
        print(f"[info] Detected {len(empty_labels)} empty masks (e.g., {empty_labels[0]})."
              f"Training can continue; review background sampling as needed.")
    if not bad_pairs:
        print("[check] Image-label filenames are consistent.")

# ---------------- Label naming and missing-label repair ----------------
def _case_id_from_image_name(p: Path) -> str:
    if p.name.endswith("_0000.nii.gz"):
        return p.name[:-len("_0000.nii.gz")]
    return p.stem

def _norm_label_name(p: Path) -> str:
    name = p.name
    if name.endswith("_0001.nii.gz"):
        return name[:-len("_0001.nii.gz")] + ".nii.gz"
    if name.endswith("_0000.nii.gz"):
        return name[:-len("_0000.nii.gz")] + ".nii.gz"
    return name

def auto_fix_labels(imagesTr: Path, labelsTr: Path, js_path: Path,
                    autofill_neg_missing: bool = True,
                    overwrite_if_conflict: bool = False):
    print("\n[fix] Normalizing labels and restoring missing negative-case masks ...")
    labelsTr.mkdir(parents=True, exist_ok=True)

    # 1) Normalize label filenames.
    renames = []
    for lp in sorted(labelsTr.glob("*.nii.gz")):
        norm = _norm_label_name(lp)
        if norm != lp.name:
            dst = labelsTr / norm
            if dst.exists():
                if overwrite_if_conflict:
                    dst.unlink()
                else:
                    print(f"[warn] Label filename conflict; retaining existing file: {dst.name}")
                    continue
            lp.rename(dst)
            renames.append((lp.name, dst.name))
    if renames:
        print(f"[fix] Normalized {len(renames)} label filenames (example): {renames[0][0]} -> {renames[0][1]}")
    else:
        print("[fix] No label filenames require normalization.")

    # 2) Recreate empty masks only for cases explicitly named _neg.
    missing_nonneg, created_neg = [], []
    for img in sorted(imagesTr.glob("*_0000.nii.gz")):
        cid = _case_id_from_image_name(img)
        lab = labelsTr / f"{cid}.nii.gz"
        if not lab.exists():
            if autofill_neg_missing and ("_neg" in cid):
                try:
                    im = nib.load(str(img))
                    shape = im.shape
                    zero = np.zeros(shape, dtype=np.uint8)
                    nl = nib.Nifti1Image(zero, affine=im.affine, header=im.header)
                    nib.save(nl, str(lab))
                    created_neg.append(lab.name)
                except Exception as e:
                    print(f"[warn] Failed to create an empty label for {lab.name}: {e}")
            else:
                missing_nonneg.append(cid)
    if created_neg:
        print(f"[fix] Created {len(created_neg)} empty labels for _neg cases (example): {created_neg[0]}")
    if missing_nonneg:
        print(f"[warn] Still {len(missing_nonneg)} non-negative cases missing labels (example): {missing_nonneg[0]}")
        print("       Positive cases normally require foreground annotations; verify case IDs and label paths (caseid.nii.gz).")

    # 3) Report empty masks without modifying them.
    zero_labels = []
    for lab in sorted(labelsTr.glob("*.nii.gz")):
        try:
            arr = nib.load(str(lab)).get_fdata()
            if float(np.max(arr)) == 0.0:
                zero_labels.append(lab.name)
        except Exception as e:
            print(f"[warn] Failed to read label {lab.name}: {e}")
    print(f"[check] Empty labels currently found in labelsTr: {len(zero_labels)} (allowed; review sampling if needed)")
    print("[fix] Label repair completed.")

# Initial dataset validation.
quick_dataset_sanity(imagesTr, labelsTr, js_path)

# Normalize labels and restore missing background-only masks.
auto_fix_labels(imagesTr, labelsTr, js_path, autofill_neg_missing=True, overwrite_if_conflict=False)

# Recheck dataset after repair.
quick_dataset_sanity(imagesTr, labelsTr, js_path)

# ---------------- Dataset fingerprint extraction ----------------
print("\n[1/4] extract_fingerprint ...")
run_cmd_pref(
    "nnUNetv2_extract_fingerprint",
    f"-d {DATASET_ID} --verify_dataset_integrity -v",
    module_fallback="nnunetv2.experiment_planning.plan_and_preprocess_entrypoints"
)

# ---------------- Experiment planning and preprocessing ----------------
print("\n[2/4] plan_and_preprocess ...")
run_cmd_pref(
    "nnUNetv2_plan_and_preprocess",
    f"-d {DATASET_ID} -c {CONFIG} -pl {PLANNER}",
    module_fallback="nnunetv2.experiment_planning.plan_and_preprocess_entrypoints"
)

# ---------------- Detect the generated plans identifier ----------------
PRE_DS = Path(PRE_ROOT) / f"Dataset{DATASET_ID}_{DATASET_NAME}"
plan_jsons = sorted(
    glob.glob(str(PRE_DS / "**" / "*Plans*.json"), recursive=True),
    key=lambda p: Path(p).stat().st_mtime, reverse=True
)
assert plan_jsons, f"No Plans*.json found in {PRE_DS}"
PLAN_NAME = Path(plan_jsons[0]).stem
print("Detected plans:", PLAN_NAME)

# ---------------- Model training with monitoring ----------------
def _fold_title(fold:int): return f"nnU-Net v2 D{DATASET_ID} ({CONFIG}) - fold {fold}"
def _fold_csv(fold:int):   return f"metrics_D{DATASET_ID}_{CONFIG}_fold{fold}.csv"

def _train_cmd(fold:int):
    cli = _exe_path("nnUNetv2_train")
    base = f'{DATASET_ID} {CONFIG} {fold} -p {PLAN_NAME}'
    if cli:
        return f'"{cli}" {base}'
    return f'"{sys.executable}" -u -m nnunetv2.run.run_training {base}'

print("\n[3/4] train + monitor (cross-val) ...")
for fold in FOLDS:
    print(f"\n===== Starting training: fold {fold} =====")
    cmd = _train_cmd(fold)
    monitor_training(run_cmd=cmd, csv_out=_fold_csv(fold), title=_fold_title(fold))
    print(f"===== Finished: fold {fold} =====")

# ---------------- Summarize recorded training metrics ----------------
def _extract_best_dice(csv_path: str):
    best = float("-inf")
    if not Path(csv_path).exists():
        return None
    with open(csv_path, "r", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            d_raw = row.get("dice")
            if not d_raw:
                continue
            try:
                d = float(d_raw)
                if d > 1: d /= 100.0
                if not math.isfinite(d):
                    continue
                best = max(best, d)
            except Exception:
                continue
    return (None if best == float("-inf") else best)

summary = {}
dice_list = []
for f in FOLDS:
    val = _extract_best_dice(_fold_csv(f))
    summary[f"fold{f}"] = (None if val is None else round(val, 6))
    if isinstance(val, float):
        dice_list.append(val)

if dice_list:
    summary["mean_best_dice"] = round(float(np.mean(dice_list)), 6)
    summary["std_best_dice"]  = round(float(np.std(dice_list)), 6)
else:
    summary["mean_best_dice"] = None
    summary["std_best_dice"]  = None

out_json = Path(RES_ROOT) / f"crossval_summary_D{DATASET_ID}_{CONFIG}.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, allow_nan=False)

print("\n[4/4] Cross-validation summary: ")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\nWritten to: {out_json}\n")


# 3. Iceberg Detection and Postprocessing
## 3.1 Single-scene SAR Iceberg Detection

In [ ]:
# -*- coding: utf-8 -*-
"""Single-scene strip-wise SAR iceberg inference using nnU-Net v2.

This script processes one GeoTIFF using nnU-Net v2, producing a prediction
raster, coastline-filtered binary mask, and iceberg polygons.
Coastline filtering removes an entire predicted polygon if it intersects
the geographic exclusion mask.

Note: If no probability output is available, this implementation uses a
binary segmentation as a proxy for probability. In that case the *_prob.tif
file does NOT contain calibrated foreground probabilities.

Set ICEBERG_PROJECT_DIR to the local project root before running.
"""

import os
import sys
import subprocess
import shutil
from pathlib import Path

import numpy as np
import nibabel as nib
import rasterio
from rasterio.windows import Window
from rasterio.merge import merge as rio_merge
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
from rasterio import features

import geopandas as gpd
from scipy.ndimage import binary_fill_holes


# ================== Input and output paths (configure before running) ==================
BASE = os.environ.get("ICEBERG_PROJECT_DIR", str(Path.cwd()))

# Specify a single input GeoTIFF.
INPUT_TIF = Path(BASE) / "predict_tif" / "Thwaites" / "202001" / "S1_202001_01.tif"

# Output directory
OUTPUT_DIR = Path(BASE) / "predict_out" / "Thwaites" / "202001"

# Polygons intersecting this geographic exclusion mask are removed in full.
COASTLINE_SHP = Path(BASE) / "mask_coastline" / "coastline_merged.shp"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ================== Processing options ==================
# True: skip if the completion marker and output rasters exist
# False: reprocess the scene even if results exist
SKIP_EXISTING = True

# True: continue with the next scene if one fails
# False: stop processing after the first failure


# ================== Temporary file options ==================
CLEAN_TEMP = True


# ================== nnU-Net inference configuration ==================
DATASET_ID = 999
CONFIG     = "2d"
PLAN_NAME  = "nnUNetResEncUNetMPlans"
TRAINER    = "nnUNetTrainer"
FOLDS      = [0]
CHECKPOINT = "checkpoint_best.pth"

USE_GPU    = True
ENABLE_TTA = False


# ================== Sliding-window and strip configuration ==================
STRIP_MAX      = 1024
STRIP_OVERLAP  = 512
TILE_SIZE      = 512
TILE_STEP      = 0.75
MASK_THRESHOLD = 0.5        # alternatively, set to 'otsu'


# ================== Binary mask postprocessing ==================
FILL_HOLES_IN_MASK = True


# ================== nnU-Net environment variables ==================
os.environ["nnUNet_raw"] = str(Path(BASE) / "nnUNet_raw_data")
os.environ["nnUNet_preprocessed"] = str(Path(BASE) / "nnUNet_preprocessed")
os.environ["nnUNet_results"] = str(Path(BASE) / "nnUNet_results")


# ==========================================================
# Utility functions
# ==========================================================

def print_step(text):
    print(f"[INFO] {text}")


def run(cmd, env=None):
    """Run a subprocess and report its standard output and errors."""
    print("\n[RUN]", cmd)

    p = subprocess.run(
        cmd,
        shell=True,
        env=env,
        capture_output=True,
        text=True
    )

    if p.stdout:
        print("[STDOUT]\n", p.stdout)

    if p.returncode != 0:
        if p.stderr:
            print("[STDERR]\n", p.stderr)

        raise subprocess.CalledProcessError(
            p.returncode,
            cmd,
            output=p.stdout,
            stderr=p.stderr
        )

    return p


def nnunet_supports(flag: str) -> bool:
    """Check whether nnUNetv2_predict supports the specified flag."""
    try:
        p = subprocess.run(
            "nnUNetv2_predict -h",
            shell=True,
            capture_output=True,
            text=True
        )

        txt = (p.stdout or "") + "\n" + (p.stderr or "")
        return flag in txt

    except Exception:
        return False


def find_model_or_die():
    """Check for a model checkpoint in the configured results directory."""
    results = os.environ.get("nnUNet_results", "")

    if not results or not os.path.isdir(results):
        print("nnUNet_results is unset or the directory does not exist: ", results)
        sys.exit(1)

    root = Path(results)
    ds_dirs = list(root.glob(f"Dataset{DATASET_ID}_*"))

    if not ds_dirs:
        print(f"Model dataset directory not found: {root}/Dataset{DATASET_ID}_*")
        sys.exit(1)

    ok = False

    for ds in ds_dirs:
        tr_dirs = list(ds.glob(f"{TRAINER}*"))

        for tr in tr_dirs:
            fold_dirs = list(tr.glob("fold_*"))

            for fd in fold_dirs:
                ck = fd / CHECKPOINT

                if ck.exists():
                    ok = True

    if not ok:
        print("Checkpoint not found. Verify the dataset directory, fold, and checkpoint name.")
        print(
            f"Expected path:  .../Dataset{DATASET_ID}_.../"
            f"{TRAINER}.../fold_*/{CHECKPOINT}"
        )
        sys.exit(1)


def affine_to_nifti(T: Affine):
    """Convert a Rasterio affine transform to a NIfTI affine matrix."""
    return np.array([
        [T.a, T.b, 0.0, T.c],
        [T.d, T.e, 0.0, T.f],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ], dtype=np.float64)


def nifti_to_rio_affine(A):
    """Convert a NIfTI affine matrix to a Rasterio affine transform."""
    return Affine(
        A[0, 0], A[0, 1], A[0, 3],
        A[1, 0], A[1, 1], A[1, 3]
    )


def iter_strips(width, height, maxdim, overlap, along="x"):
    """Generate overlapping strip windows for a raster image."""
    if along == "x":
        x0 = 0

        while x0 < width:
            x1 = min(width, x0 + maxdim)
            yield (x0, 0, x1 - x0, height)

            if x1 == width:
                break

            x0 = x1 - overlap

    else:
        y0 = 0

        while y0 < height:
            y1 = min(height, y0 + maxdim)
            yield (0, y0, width, y1 - y0)

            if y1 == height:
                break

            y0 = y1 - overlap


def mosaic_average(tile_paths, out_path, dtype):
    """Mosaic strip predictions using an arithmetic mean in overlap areas."""
    if not tile_paths:
        raise RuntimeError("No tiles to mosaic.")

    srcs = [rasterio.open(p) for p in tile_paths]

    try:
        mosaic_sum, out_T = rio_merge(srcs, method="sum")
        mosaic_cnt, _ = rio_merge(srcs, method="count")

        denom = np.maximum(mosaic_cnt, 1).astype(np.float32)
        mosaic = mosaic_sum / denom

        prof = srcs[0].profile.copy()
        prof.update(
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=out_T,
            dtype=dtype,
            compress="LZW",
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(out_path, "w", **prof) as ds:
            ds.write(mosaic[0].astype(dtype), 1)

    finally:
        for s in srcs:
            s.close()


def build_cmd(in_dir, out_dir, has_save_probs, has_tile_size, has_tile_step_size, use_tile=True):
    """Construct the nnUNetv2_predict command."""
    parts = [
        f'nnUNetv2_predict -i "{in_dir}" -o "{out_dir}"',
        f"-d {DATASET_ID} -c {CONFIG} -p {PLAN_NAME} -tr {TRAINER}",
        f"-f {' '.join(map(str, FOLDS))} -chk {CHECKPOINT}"
    ]

    if has_save_probs:
        parts.append("--save_probabilities")

    if use_tile:
        if has_tile_size:
            parts.append(f"--tile_size {TILE_SIZE}")

        if has_tile_step_size:
            parts.append(f"--tile_step_size {TILE_STEP}")

    if not ENABLE_TTA:
        parts.append("--disable_tta")

    return " ".join(parts)


def load_exclusion_mask(mask_path: Path, target_crs):
    """Load coastline/land exclusion polygons and align them to raster CRS."""
    if not mask_path.is_file():
        raise FileNotFoundError(f"Coastline mask not found: {mask_path}")
    if target_crs is None:
        raise ValueError("Input GeoTIFF has no CRS; coastline filtering is not possible.")
    mask = gpd.read_file(mask_path)
    if mask.crs is None:
        raise ValueError(f"Coastline mask has no CRS metadata: {mask_path}")
    mask = mask.to_crs(target_crs)
    mask = mask.loc[mask.geometry.notna() & ~mask.geometry.is_empty].copy()
    mask = mask.loc[mask.is_valid].copy()
    if mask.empty:
        raise ValueError(f"Coastline mask has no valid polygons: {mask_path}")
    return mask


def remove_intersecting_polygons(predictions: gpd.GeoDataFrame,
                                 exclusion: gpd.GeoDataFrame):
    """Remove entire predicted polygons intersecting the exclusion mask."""
    if predictions.empty:
        return predictions
    # Spatial-index candidates avoid a full pairwise comparison.
    candidate_pairs = exclusion.sindex.query(predictions.geometry, predicate="intersects")
    rejected = set(np.asarray(candidate_pairs[0], dtype=int).tolist())
    keep = ~np.isin(np.arange(len(predictions)), list(rejected))
    print_step(f"Coastline filter removed {len(rejected)} of {len(predictions)} polygons")
    return predictions.iloc[np.flatnonzero(keep)].copy()


def process_one_tif(
    input_tif,
    output_dir,
    has_tile_size,
    has_tile_step_size,
    has_save_probs
):
    """
    Process a single GeoTIFF scene:
    1. Divide the raster into overlapping strips.
    2. Run nnU-Net inference.
    3. Average overlapping strip outputs.
    4. Apply a segmentation threshold.
    5. Fill holes in the binary raster mask.
    6. Vectorize detected iceberg masks into a Shapefile.
    """

    input_tif = Path(input_tif)
    base1 = input_tif.stem

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_prob = output_dir / f"{base1}_prob.tif"
    output_mask = output_dir / f"{base1}_mask_filled.tif"
    output_shp  = output_dir / f"{base1}_coast_filtered.shp"
    done_marker = output_dir / f"{base1}_coast_filtered.done"

    if SKIP_EXISTING and done_marker.exists() and output_prob.exists() and output_mask.exists():
        print_step(f"Skipping {base1}: output already exists")
        return str(output_shp)

    temp_root = output_dir / f"_tmp_{base1}"
    temp_in   = temp_root / "in_strips"
    temp_pred = temp_root / "pred_strips"
    temp_tif  = temp_root / "tmp_tifs"

    if temp_root.exists():
        shutil.rmtree(temp_root)

    temp_in.mkdir(parents=True, exist_ok=True)
    temp_pred.mkdir(parents=True, exist_ok=True)
    temp_tif.mkdir(parents=True, exist_ok=True)

    try:
        print_step(f"Processing scene: {base1}")

        # --- 1. Read the source raster and create NIfTI strips ---
        print_step(f"{base1}: generating strips")

        with rasterio.open(input_tif) as src:
            H, W = src.height, src.width
            ref_crs = src.crs
            ref_prof = src.profile.copy()

            ref_prof.update(
                count=1,
                compress="LZW",
                BIGTIFF="IF_SAFER"
            )

            strips = list(iter_strips(W, H, STRIP_MAX, STRIP_OVERLAP, along="x"))

            for si, (x0, y0, w, h) in enumerate(strips):
                sub = src.read(1, window=Window(x0, y0, w, h)).astype(np.float32)
                sub_T = src.window_transform(Window(x0, y0, w, h))

                nii_path = temp_in / f"{base1}_strip{si:03d}_0000.nii.gz"

                nib.save(
                    nib.Nifti1Image(sub, affine_to_nifti(sub_T)),
                    str(nii_path)
                )

        # --- 2. Execute nnU-Net inference ---
        print_step(f"{base1}: running nnU-Net inference")

        env = os.environ.copy()

        if not USE_GPU:
            env["CUDA_VISIBLE_DEVICES"] = ""

        cmd1 = build_cmd(
            str(temp_in),
            str(temp_pred),
            has_save_probs=has_save_probs,
            has_tile_size=has_tile_size,
            has_tile_step_size=has_tile_step_size,
            use_tile=True
        )

        try:
            run(cmd1, env=env)

        except subprocess.CalledProcessError:
            print_step(f"{base1}: inference with tile flags failed; retrying without them")

            cmd2 = build_cmd(
                str(temp_in),
                str(temp_pred),
                has_save_probs=has_save_probs,
                has_tile_size=has_tile_size,
                has_tile_step_size=has_tile_step_size,
                use_tile=False
            )

            run(cmd2, env=env)

        # --- 3. Convert predicted strips to temporary georeferenced GeoTIFFs ---
        print_step(f"{base1}: aligning strip predictions")

        prob_tiles = []

        for si, (x0, y0, w, h) in enumerate(strips):
            nii_list = sorted(temp_pred.glob(f"*strip{si:03d}*.nii.gz"))

            if not nii_list:
                raise RuntimeError(f"No prediction files for strip {si}")

            seg_nii_candidates = [
                p for p in nii_list
                if "prob" not in p.name.lower()
            ]

            prob_nii_candidates = [
                p for p in nii_list
                if "prob" in p.name.lower()
            ]

            if seg_nii_candidates:
                seg_img = nib.load(str(seg_nii_candidates[0]))
            else:
                seg_img = None

            if prob_nii_candidates:
                prob_img = nib.load(str(prob_nii_candidates[0]))
            else:
                prob_img = None

            if prob_img is None:
                if seg_img is None:
                    raise RuntimeError(
                        f"Neither seg nor prob outputs present for strip {si}"
                    )

                seg_arr = (
                    np.asarray(seg_img.get_fdata(), np.float32).squeeze() != 0
                ).astype(np.uint8)

                prob_arr = seg_arr.astype(np.float32)
                A = seg_img.affine

            else:
                prob_arr = np.asarray(
                    prob_img.get_fdata(),
                    np.float32
                ).squeeze()

                A = prob_img.affine

            src_T = nifti_to_rio_affine(A)

            with rasterio.open(input_tif) as src:
                sub_T = src.window_transform(Window(x0, y0, w, h))

                prob_aligned = np.zeros((h, w), np.float32)

                reproject(
                    prob_arr,
                    prob_aligned,
                    src_transform=src_T,
                    src_crs=ref_crs,
                    dst_transform=sub_T,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )

                prob_tif = temp_tif / f"prob_strip{si:03d}.tif"

                prof_p = ref_prof.copy()
                prof_p.update(
                    nodata=None,
                    width=w,
                    height=h,
                    transform=sub_T,
                    dtype="float32",
                    compress="LZW",
                    BIGTIFF="IF_SAFER"
                )

                with rasterio.open(prob_tif, "w", **prof_p) as ds:
                    ds.write(prob_aligned, 1)

                prob_tiles.append(str(prob_tif))

        # --- 4. Average overlapping strip outputs into a full-scene mosaic ---
        print_step(f"{base1}: mosaicking prediction values")

        mosaic_average(prob_tiles, str(output_prob), "float32")

        # --- 5. Threshold, fill holes, and polygonize the mosaic ---
        print_step(f"{base1}: generating raster mask and Shapefile")

        with rasterio.open(output_prob) as srcp:
            prob_global = srcp.read(1).astype(np.float32)
            ref_transform = srcp.transform
            ref_crs = srcp.crs
            ref_prof2 = srcp.profile.copy()
            exclusion = load_exclusion_mask(COASTLINE_SHP, ref_crs)

            # Select threshold
            if isinstance(MASK_THRESHOLD, str) and MASK_THRESHOLD.lower() == "otsu":
                try:
                    from skimage.filters import threshold_otsu

                    v = prob_global[np.isfinite(prob_global)]

                    if v.size >= 16:
                        lo, hi = np.percentile(v, [1, 99])
                        thr = threshold_otsu(np.clip(v, lo, hi))
                    else:
                        thr = 0.5

                except Exception:
                    thr = 0.5

            else:
                thr = float(MASK_THRESHOLD)

            # Convert prediction values into a binary mask
            mask_global = (prob_global >= thr).astype(np.uint8)

            # Fill holes in the binary mask
            if FILL_HOLES_IN_MASK:
                mask_global = binary_fill_holes(
                    mask_global.astype(bool)
                ).astype(np.uint8)

            # Polygonize first, then remove complete iceberg candidates
            # that intersect coastline/land/ice-shelf exclusion polygons.
            shapes = [
                {"geometry": geometry, "properties": {}}
                for geometry, value in features.shapes(
                    mask_global,
                    mask=(mask_global == 1),
                    transform=ref_transform,
                )
                if value == 1
            ]

            if shapes:
                candidates = gpd.GeoDataFrame.from_features(shapes, crs=ref_crs)
                filtered = remove_intersecting_polygons(candidates, exclusion)
            else:
                filtered = gpd.GeoDataFrame(geometry=[], crs=ref_crs)

            # Save a binary mask consistent with the filtered polygons.
            if not filtered.empty:
                filtered_mask = features.rasterize(
                    ((geom, 1) for geom in filtered.geometry),
                    out_shape=mask_global.shape,
                    transform=ref_transform,
                    fill=0,
                    dtype="uint8",
                )
            else:
                filtered_mask = np.zeros_like(mask_global, dtype=np.uint8)

            ref_prof2.update(
                dtype="uint8", count=1, nodata=0,
                compress="LZW", BIGTIFF="IF_SAFER",
            )
            with rasterio.open(output_mask, "w", **ref_prof2) as ds:
                ds.write(filtered_mask, 1)

            # Remove stale Shapefile components before rewriting.
            for old_file in output_dir.glob(f"{output_shp.stem}.*"):
                if old_file.suffix.lower() != ".done" and old_file.is_file():
                    old_file.unlink()

            if not filtered.empty:
                filtered["source"] = input_tif.name
                filtered["thr"] = float(thr)
                filtered.to_file(
                    output_shp,
                    driver="ESRI Shapefile",
                    encoding="utf-8",
                )
                print_step(f"Saved {len(filtered)} coastline-filtered icebergs: {output_shp}")
            else:
                print_step("No iceberg candidates remain after coastline filtering.")

            # This marks the scene complete even when no polygons remain.
            done_marker.write_text("completed\n", encoding="utf-8")
            print_step(f"Saved coastline-filtered binary mask: {output_mask}")
            print_step("The original prediction raster is retained without coastline filtering.")

        return str(output_shp)

    finally:
        if CLEAN_TEMP and temp_root.exists():
            shutil.rmtree(temp_root)
            print_step(f"{base1}: temporary files removed")


# ==========================================================
# Single-scene entry point
# ==========================================================

if __name__ == "__main__":
    if not INPUT_TIF.is_file():
        raise FileNotFoundError(f"Input GeoTIFF does not exist: {INPUT_TIF}")
    if not COASTLINE_SHP.is_file():
        raise FileNotFoundError(f"Coastline mask does not exist: {COASTLINE_SHP}")

    print_step("Checking nnU-Net model and command-line options")
    find_model_or_die()
    process_one_tif(
        input_tif=INPUT_TIF,
        output_dir=OUTPUT_DIR,
        has_tile_size=nnunet_supports("--tile_size"),
        has_tile_step_size=nnunet_supports("--tile_step_size"),
        has_save_probs=nnunet_supports("--save_probabilities"),
    )



## 3.2 Batch SAR Iceberg Detection

In [ ]:
# -*- coding: utf-8 -*-
"""Single-scene strip-wise SAR iceberg inference using nnU-Net v2.

This script processes one GeoTIFF using nnU-Net v2, producing a prediction
raster, coastline-filtered binary mask, and iceberg polygons.
Coastline filtering removes an entire predicted polygon if it intersects
the geographic exclusion mask.

Note: If no probability output is available, this implementation uses a
binary segmentation as a proxy for probability. In that case the *_prob.tif
file does NOT contain calibrated foreground probabilities.

Set ICEBERG_PROJECT_DIR to the local project root before running.
"""

import os
import sys
import subprocess
import shutil
from pathlib import Path

import numpy as np
import nibabel as nib
import rasterio
from rasterio.windows import Window
from rasterio.merge import merge as rio_merge
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
from rasterio import features

import geopandas as gpd
from scipy.ndimage import binary_fill_holes


# ================== Input and output paths (configure before running) ==================
BASE = os.environ.get("ICEBERG_PROJECT_DIR", str(Path.cwd()))

# Each direct subfolder under INPUT_ROOT is treated as a date/scene group.
STUDY_REGION = "Getz"
INPUT_ROOT = Path(BASE) / "predict_tif" / STUDY_REGION
OUTPUT_ROOT = Path(BASE) / "predict_out" / STUDY_REGION

# Polygons intersecting this geographic exclusion mask are removed in full.
COASTLINE_SHP = Path(BASE) / "mask_coastline" / "coastline_merged.shp"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# ================== Processing options ==================
# True: skip if the completion marker and output rasters exist
# False: reprocess the scene even if results exist
SKIP_EXISTING = True

# True: continue with the next scene if one fails
# False: stop processing after the first failure
CONTINUE_ON_ERROR = True


# ================== Temporary file options ==================
CLEAN_TEMP = True


# ================== nnU-Net inference configuration ==================
DATASET_ID = 999
CONFIG     = "2d"
PLAN_NAME  = "nnUNetResEncUNetMPlans"
TRAINER    = "nnUNetTrainer"
FOLDS      = [0]
CHECKPOINT = "checkpoint_best.pth"

USE_GPU    = True
ENABLE_TTA = False


# ================== Sliding-window and strip configuration ==================
STRIP_MAX      = 1024
STRIP_OVERLAP  = 512
TILE_SIZE      = 512
TILE_STEP      = 0.75
MASK_THRESHOLD = 0.5        # alternatively, set to 'otsu'


# ================== Binary mask postprocessing ==================
FILL_HOLES_IN_MASK = True


# ================== nnU-Net environment variables ==================
os.environ["nnUNet_raw"] = str(Path(BASE) / "nnUNet_raw_data")
os.environ["nnUNet_preprocessed"] = str(Path(BASE) / "nnUNet_preprocessed")
os.environ["nnUNet_results"] = str(Path(BASE) / "nnUNet_results")


# ==========================================================
# Utility functions
# ==========================================================

def print_step(text):
    print(f"[INFO] {text}")


def run(cmd, env=None):
    """Run a subprocess and report its standard output and errors."""
    print("\n[RUN]", cmd)

    p = subprocess.run(
        cmd,
        shell=True,
        env=env,
        capture_output=True,
        text=True
    )

    if p.stdout:
        print("[STDOUT]\n", p.stdout)

    if p.returncode != 0:
        if p.stderr:
            print("[STDERR]\n", p.stderr)

        raise subprocess.CalledProcessError(
            p.returncode,
            cmd,
            output=p.stdout,
            stderr=p.stderr
        )

    return p


def nnunet_supports(flag: str) -> bool:
    """Check whether nnUNetv2_predict supports the specified flag."""
    try:
        p = subprocess.run(
            "nnUNetv2_predict -h",
            shell=True,
            capture_output=True,
            text=True
        )

        txt = (p.stdout or "") + "\n" + (p.stderr or "")
        return flag in txt

    except Exception:
        return False


def find_model_or_die():
    """Check for a model checkpoint in the configured results directory."""
    results = os.environ.get("nnUNet_results", "")

    if not results or not os.path.isdir(results):
        print("nnUNet_results is unset or the directory does not exist: ", results)
        sys.exit(1)

    root = Path(results)
    ds_dirs = list(root.glob(f"Dataset{DATASET_ID}_*"))

    if not ds_dirs:
        print(f"Model dataset directory not found: {root}/Dataset{DATASET_ID}_*")
        sys.exit(1)

    ok = False

    for ds in ds_dirs:
        tr_dirs = list(ds.glob(f"{TRAINER}*"))

        for tr in tr_dirs:
            fold_dirs = list(tr.glob("fold_*"))

            for fd in fold_dirs:
                ck = fd / CHECKPOINT

                if ck.exists():
                    ok = True

    if not ok:
        print("Checkpoint not found. Verify the dataset directory, fold, and checkpoint name.")
        print(
            f"Expected path:  .../Dataset{DATASET_ID}_.../"
            f"{TRAINER}.../fold_*/{CHECKPOINT}"
        )
        sys.exit(1)


def affine_to_nifti(T: Affine):
    """Convert a Rasterio affine transform to a NIfTI affine matrix."""
    return np.array([
        [T.a, T.b, 0.0, T.c],
        [T.d, T.e, 0.0, T.f],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ], dtype=np.float64)


def nifti_to_rio_affine(A):
    """Convert a NIfTI affine matrix to a Rasterio affine transform."""
    return Affine(
        A[0, 0], A[0, 1], A[0, 3],
        A[1, 0], A[1, 1], A[1, 3]
    )


def iter_strips(width, height, maxdim, overlap, along="x"):
    """Generate overlapping strip windows for a raster image."""
    if along == "x":
        x0 = 0

        while x0 < width:
            x1 = min(width, x0 + maxdim)
            yield (x0, 0, x1 - x0, height)

            if x1 == width:
                break

            x0 = x1 - overlap

    else:
        y0 = 0

        while y0 < height:
            y1 = min(height, y0 + maxdim)
            yield (0, y0, width, y1 - y0)

            if y1 == height:
                break

            y0 = y1 - overlap


def mosaic_average(tile_paths, out_path, dtype):
    """Mosaic strip predictions using an arithmetic mean in overlap areas."""
    if not tile_paths:
        raise RuntimeError("No tiles to mosaic.")

    srcs = [rasterio.open(p) for p in tile_paths]

    try:
        mosaic_sum, out_T = rio_merge(srcs, method="sum")
        mosaic_cnt, _ = rio_merge(srcs, method="count")

        denom = np.maximum(mosaic_cnt, 1).astype(np.float32)
        mosaic = mosaic_sum / denom

        prof = srcs[0].profile.copy()
        prof.update(
            height=mosaic.shape[1],
            width=mosaic.shape[2],
            transform=out_T,
            dtype=dtype,
            compress="LZW",
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(out_path, "w", **prof) as ds:
            ds.write(mosaic[0].astype(dtype), 1)

    finally:
        for s in srcs:
            s.close()


def build_cmd(in_dir, out_dir, has_save_probs, has_tile_size, has_tile_step_size, use_tile=True):
    """Construct the nnUNetv2_predict command."""
    parts = [
        f'nnUNetv2_predict -i "{in_dir}" -o "{out_dir}"',
        f"-d {DATASET_ID} -c {CONFIG} -p {PLAN_NAME} -tr {TRAINER}",
        f"-f {' '.join(map(str, FOLDS))} -chk {CHECKPOINT}"
    ]

    if has_save_probs:
        parts.append("--save_probabilities")

    if use_tile:
        if has_tile_size:
            parts.append(f"--tile_size {TILE_SIZE}")

        if has_tile_step_size:
            parts.append(f"--tile_step_size {TILE_STEP}")

    if not ENABLE_TTA:
        parts.append("--disable_tta")

    return " ".join(parts)


def load_exclusion_mask(mask_path: Path, target_crs):
    """Load coastline/land exclusion polygons and align them to raster CRS."""
    if not mask_path.is_file():
        raise FileNotFoundError(f"Coastline mask not found: {mask_path}")
    if target_crs is None:
        raise ValueError("Input GeoTIFF has no CRS; coastline filtering is not possible.")
    mask = gpd.read_file(mask_path)
    if mask.crs is None:
        raise ValueError(f"Coastline mask has no CRS metadata: {mask_path}")
    mask = mask.to_crs(target_crs)
    mask = mask.loc[mask.geometry.notna() & ~mask.geometry.is_empty].copy()
    mask = mask.loc[mask.is_valid].copy()
    if mask.empty:
        raise ValueError(f"Coastline mask has no valid polygons: {mask_path}")
    return mask


def remove_intersecting_polygons(predictions: gpd.GeoDataFrame,
                                 exclusion: gpd.GeoDataFrame):
    """Remove entire predicted polygons intersecting the exclusion mask."""
    if predictions.empty:
        return predictions
    # Spatial-index candidates avoid a full pairwise comparison.
    candidate_pairs = exclusion.sindex.query(predictions.geometry, predicate="intersects")
    rejected = set(np.asarray(candidate_pairs[0], dtype=int).tolist())
    keep = ~np.isin(np.arange(len(predictions)), list(rejected))
    print_step(f"Coastline filter removed {len(rejected)} of {len(predictions)} polygons")
    return predictions.iloc[np.flatnonzero(keep)].copy()


def scene_is_complete(input_tif: Path, output_dir: Path) -> bool:
    """Require the two output rasters and the explicit completion marker."""
    stem = Path(input_tif).stem
    return all((
        (output_dir / f"{stem}_prob.tif").is_file(),
        (output_dir / f"{stem}_mask_filled.tif").is_file(),
        (output_dir / f"{stem}_coast_filtered.done").is_file(),
    ))


def folder_is_complete(input_dir: Path, output_dir: Path) -> bool:
    """Return True only when every GeoTIFF in a nonempty folder is complete."""
    rasters = sorted(set(input_dir.glob("*.tif")) | set(input_dir.glob("*.tiff")))
    return bool(rasters) and all(scene_is_complete(p, output_dir) for p in rasters)


def process_one_tif(
    input_tif,
    output_dir,
    has_tile_size,
    has_tile_step_size,
    has_save_probs
):
    """
    Process a single GeoTIFF scene:
    1. Divide the raster into overlapping strips.
    2. Run nnU-Net inference.
    3. Average overlapping strip outputs.
    4. Apply a segmentation threshold.
    5. Fill holes in the binary raster mask.
    6. Vectorize detected iceberg masks into a Shapefile.
    """

    input_tif = Path(input_tif)
    base1 = input_tif.stem

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_prob = output_dir / f"{base1}_prob.tif"
    output_mask = output_dir / f"{base1}_mask_filled.tif"
    output_shp  = output_dir / f"{base1}_coast_filtered.shp"
    done_marker = output_dir / f"{base1}_coast_filtered.done"

    if SKIP_EXISTING and scene_is_complete(input_tif, output_dir):
        print_step(f"Skipping {base1}: output already exists")
        return str(output_shp)

    temp_root = output_dir / f"_tmp_{base1}"
    temp_in   = temp_root / "in_strips"
    temp_pred = temp_root / "pred_strips"
    temp_tif  = temp_root / "tmp_tifs"

    if temp_root.exists():
        shutil.rmtree(temp_root)

    temp_in.mkdir(parents=True, exist_ok=True)
    temp_pred.mkdir(parents=True, exist_ok=True)
    temp_tif.mkdir(parents=True, exist_ok=True)

    try:
        print_step(f"Processing scene: {base1}")

        # --- 1. Read the source raster and create NIfTI strips ---
        print_step(f"{base1}: generating strips")

        with rasterio.open(input_tif) as src:
            H, W = src.height, src.width
            ref_crs = src.crs
            ref_prof = src.profile.copy()

            ref_prof.update(
                count=1,
                compress="LZW",
                BIGTIFF="IF_SAFER"
            )

            strips = list(iter_strips(W, H, STRIP_MAX, STRIP_OVERLAP, along="x"))

            for si, (x0, y0, w, h) in enumerate(strips):
                sub = src.read(1, window=Window(x0, y0, w, h)).astype(np.float32)
                sub_T = src.window_transform(Window(x0, y0, w, h))

                nii_path = temp_in / f"{base1}_strip{si:03d}_0000.nii.gz"

                nib.save(
                    nib.Nifti1Image(sub, affine_to_nifti(sub_T)),
                    str(nii_path)
                )

        # --- 2. Execute nnU-Net inference ---
        print_step(f"{base1}: running nnU-Net inference")

        env = os.environ.copy()

        if not USE_GPU:
            env["CUDA_VISIBLE_DEVICES"] = ""

        cmd1 = build_cmd(
            str(temp_in),
            str(temp_pred),
            has_save_probs=has_save_probs,
            has_tile_size=has_tile_size,
            has_tile_step_size=has_tile_step_size,
            use_tile=True
        )

        try:
            run(cmd1, env=env)

        except subprocess.CalledProcessError:
            print_step(f"{base1}: inference with tile flags failed; retrying without them")

            cmd2 = build_cmd(
                str(temp_in),
                str(temp_pred),
                has_save_probs=has_save_probs,
                has_tile_size=has_tile_size,
                has_tile_step_size=has_tile_step_size,
                use_tile=False
            )

            run(cmd2, env=env)

        # --- 3. Convert predicted strips to temporary georeferenced GeoTIFFs ---
        print_step(f"{base1}: aligning strip predictions")

        prob_tiles = []

        for si, (x0, y0, w, h) in enumerate(strips):
            nii_list = sorted(temp_pred.glob(f"*strip{si:03d}*.nii.gz"))

            if not nii_list:
                raise RuntimeError(f"No prediction files for strip {si}")

            seg_nii_candidates = [
                p for p in nii_list
                if "prob" not in p.name.lower()
            ]

            prob_nii_candidates = [
                p for p in nii_list
                if "prob" in p.name.lower()
            ]

            if seg_nii_candidates:
                seg_img = nib.load(str(seg_nii_candidates[0]))
            else:
                seg_img = None

            if prob_nii_candidates:
                prob_img = nib.load(str(prob_nii_candidates[0]))
            else:
                prob_img = None

            if prob_img is None:
                if seg_img is None:
                    raise RuntimeError(
                        f"Neither seg nor prob outputs present for strip {si}"
                    )

                seg_arr = (
                    np.asarray(seg_img.get_fdata(), np.float32).squeeze() != 0
                ).astype(np.uint8)

                prob_arr = seg_arr.astype(np.float32)
                A = seg_img.affine

            else:
                prob_arr = np.asarray(
                    prob_img.get_fdata(),
                    np.float32
                ).squeeze()

                A = prob_img.affine

            src_T = nifti_to_rio_affine(A)

            with rasterio.open(input_tif) as src:
                sub_T = src.window_transform(Window(x0, y0, w, h))

                prob_aligned = np.zeros((h, w), np.float32)

                reproject(
                    prob_arr,
                    prob_aligned,
                    src_transform=src_T,
                    src_crs=ref_crs,
                    dst_transform=sub_T,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )

                prob_tif = temp_tif / f"prob_strip{si:03d}.tif"

                prof_p = ref_prof.copy()
                prof_p.update(
                    nodata=None,
                    width=w,
                    height=h,
                    transform=sub_T,
                    dtype="float32",
                    compress="LZW",
                    BIGTIFF="IF_SAFER"
                )

                with rasterio.open(prob_tif, "w", **prof_p) as ds:
                    ds.write(prob_aligned, 1)

                prob_tiles.append(str(prob_tif))

        # --- 4. Average overlapping strip outputs into a full-scene mosaic ---
        print_step(f"{base1}: mosaicking prediction values")

        mosaic_average(prob_tiles, str(output_prob), "float32")

        # --- 5. Threshold, fill holes, and polygonize the mosaic ---
        print_step(f"{base1}: generating raster mask and Shapefile")

        with rasterio.open(output_prob) as srcp:
            prob_global = srcp.read(1).astype(np.float32)
            ref_transform = srcp.transform
            ref_crs = srcp.crs
            ref_prof2 = srcp.profile.copy()
            exclusion = load_exclusion_mask(COASTLINE_SHP, ref_crs)

            # Select threshold
            if isinstance(MASK_THRESHOLD, str) and MASK_THRESHOLD.lower() == "otsu":
                try:
                    from skimage.filters import threshold_otsu

                    v = prob_global[np.isfinite(prob_global)]

                    if v.size >= 16:
                        lo, hi = np.percentile(v, [1, 99])
                        thr = threshold_otsu(np.clip(v, lo, hi))
                    else:
                        thr = 0.5

                except Exception:
                    thr = 0.5

            else:
                thr = float(MASK_THRESHOLD)

            # Convert prediction values into a binary mask
            mask_global = (prob_global >= thr).astype(np.uint8)

            # Fill holes in the binary mask
            if FILL_HOLES_IN_MASK:
                mask_global = binary_fill_holes(
                    mask_global.astype(bool)
                ).astype(np.uint8)

            # Polygonize first, then remove complete iceberg candidates
            # that intersect coastline/land/ice-shelf exclusion polygons.
            shapes = [
                {"geometry": geometry, "properties": {}}
                for geometry, value in features.shapes(
                    mask_global,
                    mask=(mask_global == 1),
                    transform=ref_transform,
                )
                if value == 1
            ]

            if shapes:
                candidates = gpd.GeoDataFrame.from_features(shapes, crs=ref_crs)
                filtered = remove_intersecting_polygons(candidates, exclusion)
            else:
                filtered = gpd.GeoDataFrame(geometry=[], crs=ref_crs)

            # Save a binary mask consistent with the filtered polygons.
            if not filtered.empty:
                filtered_mask = features.rasterize(
                    ((geom, 1) for geom in filtered.geometry),
                    out_shape=mask_global.shape,
                    transform=ref_transform,
                    fill=0,
                    dtype="uint8",
                )
            else:
                filtered_mask = np.zeros_like(mask_global, dtype=np.uint8)

            ref_prof2.update(
                dtype="uint8", count=1, nodata=0,
                compress="LZW", BIGTIFF="IF_SAFER",
            )
            with rasterio.open(output_mask, "w", **ref_prof2) as ds:
                ds.write(filtered_mask, 1)

            # Remove stale Shapefile components before rewriting.
            for old_file in output_dir.glob(f"{output_shp.stem}.*"):
                if old_file.suffix.lower() != ".done" and old_file.is_file():
                    old_file.unlink()

            if not filtered.empty:
                filtered["source"] = input_tif.name
                filtered["thr"] = float(thr)
                filtered.to_file(
                    output_shp,
                    driver="ESRI Shapefile",
                    encoding="utf-8",
                )
                print_step(f"Saved {len(filtered)} coastline-filtered icebergs: {output_shp}")
            else:
                print_step("No iceberg candidates remain after coastline filtering.")

            # This marks the scene complete even when no polygons remain.
            done_marker.write_text("completed\n", encoding="utf-8")
            print_step(f"Saved coastline-filtered binary mask: {output_mask}")
            print_step("The original prediction raster is retained without coastline filtering.")

        return str(output_shp)

    finally:
        if CLEAN_TEMP and temp_root.exists():
            shutil.rmtree(temp_root)
            print_step(f"{base1}: temporary files removed")


# ==========================================================
# Batch processing entry point
# ==========================================================

def main() -> None:
    """Process all scene rasters grouped by immediate date subfolders."""
    if not INPUT_ROOT.is_dir():
        raise FileNotFoundError(f"Input root does not exist: {INPUT_ROOT}")
    if not COASTLINE_SHP.is_file():
        raise FileNotFoundError(f"Coastline mask does not exist: {COASTLINE_SHP}")

    find_model_or_die()
    has_tile_size = nnunet_supports("--tile_size")
    has_tile_step_size = nnunet_supports("--tile_step_size")
    has_save_probs = nnunet_supports("--save_probabilities")

    date_folders = sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir())
    if not date_folders:
        raise RuntimeError(f"No subdirectories found under: {INPUT_ROOT}")

    n_processed = n_skipped = n_failed = n_empty = 0
    for group_index, input_dir in enumerate(date_folders, 1):
        output_dir = OUTPUT_ROOT / input_dir.name
        rasters = sorted(set(input_dir.glob("*.tif")) | set(input_dir.glob("*.tiff")))
        print_step(f"Folder {group_index}/{len(date_folders)}: {input_dir.name}")
        if not rasters:
            print_step(f"Skipping empty folder: {input_dir.name}")
            n_empty += 1
            continue
        if SKIP_EXISTING and folder_is_complete(input_dir, output_dir):
            print_step(f"Skipping completed folder: {input_dir.name}")
            n_skipped += len(rasters)
            continue
        output_dir.mkdir(parents=True, exist_ok=True)

        for scene_index, tif_path in enumerate(rasters, 1):
            print_step(f"Scene {scene_index}/{len(rasters)}: {tif_path.name}")
            if SKIP_EXISTING and scene_is_complete(tif_path, output_dir):
                n_skipped += 1
                continue
            try:
                process_one_tif(
                    input_tif=tif_path,
                    output_dir=output_dir,
                    has_tile_size=has_tile_size,
                    has_tile_step_size=has_tile_step_size,
                    has_save_probs=has_save_probs,
                )
                n_processed += 1
            except Exception as exc:
                n_failed += 1
                print_step(f"[ERROR] {tif_path.name}: {exc}")
                if not CONTINUE_ON_ERROR:
                    raise

    print_step(
        f"Batch completed. New: {n_processed}; skipped: {n_skipped}; "
        f"failed: {n_failed}; empty folders: {n_empty}."
    )


if __name__ == "__main__":
    main()
